# Baseline 3D-patch — W2 ngày 5

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

Train DenseNet121-3D (8 pha ghép kênh → 7 lớp) trên **cache đã tiền xử lý**, một fold,
để lấy **số mốc đầu tiên** của dự án. Chưa cần CI — bootstrap CI là việc W3.

**Thứ tự:**
1. Bootstrap (clone code + tìm cache đã mount)
2. Smoke test: nhãn, augment, model forward
3. **Kiểm tra tỉnh táo**: model có học nổi 8 mẫu không (cổng chặn, ~90 giây)
4. Train fold 1
5. Đọc số + ma trận nhầm lẫn

⚠️ Notebook **không được chạm test-104** (AGENTS.md §3.4). Ở đây chỉ có train/val fold.

⚠️ Mọi thứ tốn GPU đều nằm sau các cổng chặn rẻ tiền. Nếu một cổng dừng lại, đọc
thông báo của nó — nó nói đúng chỗ cần sửa, và sửa xong rẻ hơn nhiều so với một run
20 phút cho ra số vô nghĩa.

⚠️ Kaggle cắt session bất cứ lúc nào. Chạy lại cell train sẽ **resume** từ `last.pt`.

## 0. Bootstrap

Luôn xoá + clone lại, **và xoá module `src.*` đã nạp trong bộ nhớ**. Clone lại code
không tự làm việc thứ hai: Python giữ bản đã import trong `sys.modules`, nên bỏ qua
bước đó thì `repo commit` in ra bản mới còn code chạy vẫn là bản cũ.

Hai dòng đầu output là bằng chứng: `repo commit` (bản nào) và `code đang dùng`
(nạp từ đâu). Nếu vẫn gặp lỗi cũ sau khi đã sửa code → **Restart kernel**, rồi chạy lại.

Cache lấy từ Kaggle Dataset `marcohoang/lld-mmri-3` (Private, xem `configs/preprocess.yaml`).
Không đoán đường dẫn mount — tìm thư mục nào chứa `cache_meta.json`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"
ON_KAGGLE = Path("/kaggle/input").exists()

if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    subprocess.run(["rm", "-rf", str(REPO)], check=False)
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    sys.path.insert(0, str(REPO))
    os.environ["LLDMMRI_OUTPUT_DIR"] = "/kaggle/working/runs/baseline_3dpatch"
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(REPO))

# Xoá module src.* đã nạp từ lần chạy trước. Clone lại code KHÔNG tự làm điều này:
# Python giữ nguyên bản đã import trong sys.modules, nên nếu bỏ qua bước này thì
# dòng "repo commit" bên dưới in ra bản mới trong khi code đang chạy vẫn là bản cũ
# — đúng cái bẫy đã mất một phiên để tìm ra (WORKLOG S-035).
for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

if ON_KAGGLE:
    # --no-deps: pip TUYỆT ĐỐI không được đụng tới torch/numpy có sẵn của Kaggle —
    # nâng/hạ torch ở đây vừa lâu vừa dễ làm hỏng CUDA của cả session.
    # Cố ý KHÔNG pin version: pin 1.3.2 (như requirements.txt cho máy local) có thể
    # không import được với torch mới của Kaggle, và một run chết vì lý do đó tốn hơn
    # là mất tính pin. Bù lại, version thật được IN RA ở cell sau và ghi vào WORKLOG.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True
    )

from src.utils.io import (  # noqa: E402
    describe_tree,
    find_cache_dir,
    load_yaml,
    repo_root,
    resolve_cache_dir,
)

# Bằng chứng code đang chạy đúng là bản vừa clone, không phải bản cũ trong bộ nhớ.
print("code đang dùng:", repo_root())
assert repo_root() == REPO.resolve(), (
    f"module src/ đang nạp từ {repo_root()} chứ không phải {REPO}. "
    "Restart kernel rồi chạy lại từ cell này."
)

if ON_KAGGLE:
    cache_dir = find_cache_dir(["/kaggle/input"])
    if cache_dir is None:
        print("KHÔNG THẤY CACHE. Đang mount những thứ này dưới /kaggle/input:")
        for line in describe_tree("/kaggle/input"):
            print("   ", line)
        raise RuntimeError(
            "Chưa có cache. Panel phải -> Input -> Add Input -> tab Datasets -> "
            "'Your Datasets' -> lld-mmri-3 (Private, version 1). "
            "Nếu danh sách in ở trên ĐÃ có cache mà vẫn lỗi thì gửi nguyên đoạn đó."
        )
    os.environ["LLDMMRI_CACHE_DIR"] = str(cache_dir)

CFG_PATH = REPO / "configs" / "baseline_3dpatch.yaml"
CFG = load_yaml(CFG_PATH)
CACHE_DIR = resolve_cache_dir(CFG)
SPLITS_DIR = repo_root() / CFG.get("splits_dir", "splits")

n_npz = len(list(CACHE_DIR.glob("*.npz")))
meta_path = CACHE_DIR / "cache_meta.json"
print("cache dir :", CACHE_DIR)
print("file .npz :", n_npz, "(cần 498)")
print("splits    :", SPLITS_DIR, "| có labels_trainval.txt:",
      (SPLITS_DIR / "labels_trainval.txt").is_file())
print("meta      :", meta_path.read_text(encoding="utf-8")[:400] if meta_path.exists()
      else "KHÔNG CÓ cache_meta.json — không xác minh được cache dựng bằng config nào")
if n_npz != 498:
    print(f"\n⚠️  Chỉ thấy {n_npz}/498 file. Dừng lại kiểm tra trước khi train.")

## 1. Smoke test — dataset + model

Rẻ và chạy trước khi tốn GPU: batch phải ra đúng `[B, 8, 96, 96, 48]`, model phải
trả logits `[B, 7]`. Cũng in phân bố lớp của fold — để biết lớp hiếm hiếm tới mức nào.

In [ ]:
from collections import Counter

import monai
import torch
from torch.utils.data import DataLoader

from src.data.dataset import build_fold_datasets, find_label_mismatches
from src.data.taxonomy import SHORT_NAMES
from src.data.transforms import build_train_transform
from src.models import build_model, count_parameters
from src.train.loop import class_weights_from_labels

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| monai", monai.__version__)
print("device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

FOLD = CFG["fold"]
# Không truyền train_transform ở đây: cell này cần dữ liệu thô để kiểm augment riêng.
train_ds, val_ds = build_fold_datasets(CACHE_DIR, FOLD, splits_dir=SPLITS_DIR)
print(f"\nfold {FOLD}: train={len(train_ds)} val={len(val_ds)} (tổng phải = 394)")

# --- 1. Nhãn cache có khớp nhãn splits/ không -------------------------------
# Lệch nhãn KHÔNG làm train báo lỗi: loss vẫn giảm, metric vẫn ra số, chỉ là mọi
# kết quả đều vô nghĩa. Vài giây ở đây, thay vì phát hiện sau khi đã train xong.
bad = find_label_mismatches(train_ds) + find_label_mismatches(val_ds)
print("nhãn cache vs splits:", "KHỚP TOÀN BỘ" if not bad else f"LỆCH {len(bad)} ca -> {bad[:5]}")
assert not bad, "cache và splits không cùng một nguồn — dừng lại"

# --- 2. Phân bố lớp + trọng số ----------------------------------------------
train_labels = [label for _, label, _ in train_ds.samples]
counts = Counter(train_labels)
weights = class_weights_from_labels(train_labels)
for k in sorted(SHORT_NAMES):
    print(f"  {SHORT_NAMES[k]:>7}: {counts.get(k, 0):3d} ca | trọng số {weights[k]:.2f}")

# --- 3. Augmentation chạy thật ----------------------------------------------
augment = build_train_transform(CFG["data"].get("augment"))
assert augment is not None, "config có khối augment nhưng build_train_transform trả None"
raw = train_ds[0]["image"]
before = tuple(raw.shape)
for _ in range(20):
    out = augment({"image": raw.clone()})["image"]
    assert tuple(out.shape) == before, f"augment đổi shape {before} -> {tuple(out.shape)}"
    assert torch.isfinite(out).all(), "augment sinh NaN/Inf"
print(f"\naugment: 20 lần, shape giữ nguyên {before}, không NaN/Inf")

# --- 4. Model: shape + norm layer THỰC SỰ được dựng -------------------------
# In ra loại norm thật thay vì tin vào config: `norm` đi qua factory của MONAI, và
# một chuỗi sai chính tả sẽ lặng lẽ rơi về mặc định.
model = build_model(CFG["model"]).to(DEVICE)
model.eval()
batch = next(iter(DataLoader(train_ds, batch_size=2, shuffle=False)))
with torch.no_grad():
    logits = model(batch["image"].to(DEVICE))

norm_layers = [m for m in model.modules() if "Norm" in type(m).__name__]
kinds = Counter(type(m).__name__ for m in norm_layers)
affine = {getattr(m, "affine", None) for m in norm_layers}
print("batch      :", tuple(batch["image"].shape), batch["image"].dtype)
print("logits     :", tuple(logits.shape), "| finite:", bool(torch.isfinite(logits).all()))
print("tham số    :", f"{count_parameters(model):,}")
print("config norm:", CFG["model"]["norm"])
print("norm thật  :", dict(kinds), "| affine:", affine)

assert tuple(logits.shape) == (2, 7)
assert affine == {True}, "norm không affine = mất scale/shift học được ở mọi lớp"
# Loại norm dựng ra phải khớp thứ config nói — bắt lỗi chính tả rơi về mặc định.
expected_kind = {
    "batch": "BatchNorm3d", "instance": "InstanceNorm3d", "group": "GroupNorm",
}[CFG["model"]["norm"] if isinstance(CFG["model"]["norm"], str) else CFG["model"]["norm"][0]]
assert set(kinds) == {expected_kind}, f"config nói {expected_kind}, model dựng ra {dict(kinds)}"

del model, logits, batch
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()  # trả VRAM lại cho cell sau

## 1b. Kiểm tra tỉnh táo — model có học nổi 8 mẫu không? ⚠️ CỔNG CHẶN

Nhồi 8 mẫu (trải nhiều lớp) vào model vài chục bước. Một kiến trúc lành mạnh phải
**thuộc lòng** chúng: loss → ~0, accuracy → 1.0. Không làm nổi việc đó thì train 60
epoch chỉ tốn thời gian để khẳng định lại điều đã biết.

Chạy cho **cả ba** lựa chọn `norm` để so bằng bằng chứng, không bằng phỏng đoán.
Tốn ~30 giây mỗi phương án — rẻ hơn nhiều so với một run 20 phút hỏng.

*Vì sao có cell này:* bản InstanceNorm sập về đoán một lớp duy nhất (macro-F1 0.0668
đứng yên, train loss = ln 7), và phải mất trọn một run mới nhìn ra — WORKLOG S-039.

In [ ]:
import gc

from src.models.densenet3d import normalize_norm_spec
from src.train.sanity import overfit_check, verdict
from src.utils.seed import set_seed

CANDIDATES = {
    "batch": "batch",
    "instance+affine": ["instance", {"affine": True}],
    "group(8)": ["group", {"num_groups": 8, "affine": True}],
}

results = {}
for label, norm_spec in CANDIDATES.items():
    set_seed(CFG["seed"])  # cùng khởi tạo -> so được
    candidate = build_model({**CFG["model"], "norm": norm_spec})
    results[label] = overfit_check(train_ds, candidate, DEVICE, n_samples=8, passes=40)
    del candidate
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print(f"{'norm':<16} {'loss đầu':>9} {'loss cuối':>10} {'acc':>6}  kết luận")
print("-" * 56)
for label, r in results.items():
    print(
        f"{label:<16} {r['loss_start']:>9.3f} {r['loss_end']:>10.3f} "
        f"{r['accuracy_end']:>6.2f}  {verdict(r)}"
    )
print(f"\n(8 mẫu trải {int(next(iter(results.values()))['n_classes'])} lớp; "
      "loss của đoán ngẫu nhiên = ln 7 = 1.946)")

# Cổng chặn: norm trong config phải là một phương án HỌC ĐƯỢC. Config vẫn là nguồn
# sự thật duy nhất (AGENTS.md §8) — cell này KHÔNG tự sửa nó, chỉ từ chối chạy tiếp.
current = normalize_norm_spec(CFG["model"]["norm"])
matching = [
    label for label, spec in CANDIDATES.items() if normalize_norm_spec(spec) == current
]
if matching and verdict(results[matching[0]]) == "SẬP":
    raise RuntimeError(
        f"norm trong config ({matching[0]}) không học nổi 8 mẫu. "
        f"Sửa configs/baseline_3dpatch.yaml sang phương án HỌC ĐƯỢC rồi push, "
        f"đừng tốn 20 phút GPU cho nó."
    )

## 1c. Đo thời gian mỗi epoch ⚠️ CỔNG CHẶN

Recipe official là **300 epoch**. Trước khi cam kết một run dài như vậy, đo thật 2 epoch
rồi ngoại suy — nếu tổng vượt ngân sách thì biết **ngay bây giờ**, không phải sau 3 tiếng.

Rủi ro cụ thể đang đo: augmentation mới có `scipy.ndimage.rotate` chạy trên **CPU** trong
DataLoader worker, trên khối `[8,96,96,48]` lớn gấp 2,5 lần khối của official. Đây là
thứ chưa từng được đo, và nó nằm trên đường tới hạn của thời gian mỗi epoch.

Nếu cell này báo vượt ngân sách: tăng `num_workers`, hoặc hạ `rotate_order`, hoặc giảm
`epochs` — nhưng ghi rõ vào WORKLOG là đã lệch khỏi recipe official.

In [ ]:
import gc
import time

from src.train.loop import make_amp_scaler, run_epoch
from src.train.run import build_loaders, build_param_groups, build_scheduler
from src.utils.seed import set_seed

# Ngân sách lấy từ ràng buộc THẬT, không phải con số cho đẹp:
#   - Kaggle cắt session ở 12h (AGENTS.md §7) -> một fold phải vừa, có dư.
#   - Quota GPU ~30h/tuần -> 5 fold phải nằm gọn trong đó.
BUDGET_HOURS_PER_FOLD = 6.0
BUDGET_HOURS_ALL_FOLDS = 25.0
N_FOLDS = 5
PROBE_EPOCHS = 2

TCFG, DCFG = CFG["train"], CFG["data"]
set_seed(CFG["seed"])

# Dùng ĐÚNG hàm mà train thật dùng. Probe tự dựng loader theo cách khác thì con số
# đo được không dự đoán được run thật — mà đó là toàn bộ mục đích của cell này.
probe_train_loader, probe_val_loader, probe_labels = build_loaders(CFG, FOLD)

probe_model = build_model(CFG["model"]).to(DEVICE)
probe_opt = torch.optim.AdamW(
    build_param_groups(probe_model, float(TCFG["weight_decay"])), lr=float(TCFG["lr"])
)
probe_sched = build_scheduler(probe_opt, TCFG, int(TCFG["epochs"]))
probe_amp = bool(TCFG.get("amp", True)) and DEVICE.type == "cuda"
probe_scaler = make_amp_scaler(probe_amp)
criterion = torch.nn.CrossEntropyLoss()

accum = int(TCFG["accum_steps"])
print(f"batch hiệu dụng {int(DCFG['batch_size']) * accum} · "
      f"~{len(probe_labels) // (int(DCFG['batch_size']) * accum)} bước/epoch")
print(f"lr {TCFG['lr']} · wd {TCFG['weight_decay']} · warmup {TCFG['warmup_epochs']} epoch")
print(f"workers {DCFG['num_workers']} · persistent {DCFG.get('persistent_workers')} · "
      f"prefetch {DCFG.get('prefetch_factor')}")
print(f"augment: {DCFG.get('augment')}\n")

timings = []
for i in range(PROBE_EPOCHS):
    t0 = time.time()
    tr = run_epoch(probe_model, probe_train_loader, DEVICE, criterion,
                   optimizer=probe_opt, scaler=probe_scaler, accum_steps=accum, amp=probe_amp)
    va = run_epoch(probe_model, probe_val_loader, DEVICE, criterion, amp=probe_amp)
    probe_sched.step()
    timings.append(time.time() - t0)
    print(f"epoch thử {i + 1}: {timings[-1]:.1f}s | train {tr['loss']:.4f} | val {va['loss']:.4f} "
          f"| lr {probe_opt.param_groups[0]['lr']:.2e}")

# Epoch đầu gánh chi phí khởi động worker -> lấy epoch sau làm ước lượng.
per_epoch = timings[-1]
hours_one = per_epoch * int(TCFG["epochs"]) / 3600
hours_all = hours_one * N_FOLDS
print(f"\n~{per_epoch:.1f}s/epoch × {TCFG['epochs']} epoch = **{hours_one:.2f} giờ/fold**")
print(f"{N_FOLDS} fold = {hours_all:.1f} giờ (quota GPU ~30h/tuần)")

del probe_model, probe_opt, probe_sched, probe_scaler
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

if hours_one > BUDGET_HOURS_PER_FOLD:
    raise RuntimeError(
        f"{hours_one:.2f} giờ/fold, vượt ngân sách {BUDGET_HOURS_PER_FOLD} giờ.\n"
        "Xử lý theo thứ tự ƯU TIÊN KỸ THUẬT TRƯỚC (không đụng recipe): tăng num_workers, "
        "bật persistent_workers, tăng prefetch_factor. Chỉ khi hết cách mới hạ "
        "rotate_order hoặc giảm epochs — và phải ghi WORKLOG là đã lệch recipe official."
    )
if hours_all > BUDGET_HOURS_ALL_FOLDS:
    print(f"\n⚠️  Một fold thì ổn, nhưng {N_FOLDS} fold = {hours_all:.1f} giờ, sát quota tuần. "
          "Chạy fold 1 trước và quyết theo luật ở WORKLOG S-043 rồi mới chạy tiếp.")
print(f"\nTrong ngân sách {BUDGET_HOURS_PER_FOLD} giờ/fold — chạy được.")

## 2. Train fold 1

Toàn bộ hyperparam nằm trong `configs/baseline_3dpatch.yaml` — sửa ở đó rồi commit,
**đừng sửa trong notebook** (số báo cáo phải tái lập được từ config + seed).

Chỉ chạy cell này khi mục 1b đã cho thấy `norm` trong config **học được**.

Nếu session chết giữa chừng: chạy lại đúng cell này, nó tự tiếp từ epoch dở dang.
Mỗi kiến trúc ghi vào thư mục riêng theo hash, nên checkpoint của cấu hình khác
không bao giờ lẫn vào đây.

In [ ]:
from src.train.run import train

result = train(CFG_PATH, fold_override=FOLD)
print(result)

## 3. Số mốc + ma trận nhầm lẫn

Đây là **số val của 1 fold, 1 seed** — mốc để so, không phải kết quả báo cáo.
Kết quả báo cáo cần CV 5-fold + bootstrap CI (W3, AGENTS.md §3.5).

In [ ]:
import json

import numpy as np

from src.train.run import run_dir

# Mỗi kiến trúc có thư mục riêng theo hash khối `model:` — nhờ vậy kết quả của bản
# BatchNorm cũ và bản InstanceNorm nằm cạnh nhau, so được, không đè nhau.
RUN_DIR = run_dir(CFG, FOLD)
print("run dir:", RUN_DIR)
best = json.loads((RUN_DIR / "metrics_best.json").read_text(encoding="utf-8"))

print(f"\nfold {best['fold']} · epoch {best['epoch']} · seed {best['seed']}")
for key in ("macro_f1", "balanced_accuracy", "accuracy", "cohen_kappa"):
    print(f"  {key:>18}: {best[key]:.4f}")

print("\nF1 từng lớp:")
for k, f1 in enumerate(best["per_class_f1"]):
    print(f"  {SHORT_NAMES[k]:>7}: {f1:.3f}")

print("\nMa trận nhầm lẫn (hàng = thật, cột = đoán):")
matrix = np.array(best["confusion_matrix"])
header = "        " + "".join(f"{SHORT_NAMES[k]:>8}" for k in sorted(SHORT_NAMES))
print(header)
for k, row in enumerate(matrix):
    print(f"{SHORT_NAMES[k]:>7} " + "".join(f"{v:>8d}" for v in row))

print("\n--- train_log.csv (10 epoch cuối) ---")
print("".join((RUN_DIR / "train_log.csv").read_text(encoding="utf-8").splitlines(True)[-10:]))

# Mốc để so: bản BatchNorm, cùng fold/seed (WORKLOG S-036).
print(f"mốc cũ (BatchNorm): macro-F1 0.2725 @ epoch 11")
print(f"lần này            : macro-F1 {best['macro_f1']:.4f} @ epoch {best['epoch']}")

## 4. Giữ lại gì

`/kaggle/working/runs/...` biến mất khi session kết thúc → **tải về hoặc save output**:

- `metrics_best.json`, `train_log.csv`, `config_used.json` → chép số vào WORKLOG.
- `val_probs_best.npz` → W3 tính bootstrap CI, W5 tính calibration/selective **mà không train lại**.
- `best.pt`, `last.pt` → checkpoint. **Không commit vào git** (AGENTS.md §3.10).